In [1]:
print("hello")

hello


In [2]:
import pandas as pd

In [3]:
import pandas as pd

ts_data = pd.read_parquet('../data/transformed/ts_data_2023_01.parquet')

ts_data.head()

,pickup_hour,rides,user_location_id
0,2023-01-01 00:00:00,19,4
1,2023-01-01 01:00:00,28,4
2,2023-01-01 02:00:00,43,4
3,2023-01-01 03:00:00,33,4
4,2023-01-01 04:00:00,12,4


In [4]:
ts_data_one_location = ts_data[
    ts_data['user_location_id'] == 43
].reset_index(drop=True)

In [5]:
def get_cutoff_indices(data, n_features, step_size):
    stop_position = len(data) - 1

    subseq_first_idx = 0
    subseq_mid_idx = n_features
    subseq_last_idx = n_features + 1

    indices = []

    while subseq_last_idx <= stop_position:
        indices.append((subseq_first_idx, subseq_mid_idx, subseq_last_idx))

        subseq_first_idx += step_size
        subseq_mid_idx += step_size
        subseq_last_idx += step_size

    return indices

In [6]:
cutoff_indices = get_cutoff_indices(
    ts_data_one_location,
    n_features=24,   # last 24 hours
    step_size=1
)

In [7]:
X = []
y = []

for start, mid, end in cutoff_indices:
    X.append(ts_data_one_location['rides'].iloc[start:mid].values)
    y.append(ts_data_one_location['rides'].iloc[mid:end].values)

import numpy as np

X = np.array(X)
y = np.array(y)

In [8]:
print(X.shape)
print(y.shape)

(719, 24)
(719, 1)


In [ ]:
indices = get_cutoff_indices(
    ts_data_one_location,
    n_features=24,
    step_size=1
)

In [10]:
indices[:5]

[(0, 24, 25), (1, 25, 26), (2, 26, 27), (3, 27, 28), (4, 28, 29)]

In [12]:
n_features = 24
step_size = 1

indices = get_cutoff_indices(
    ts_data_one_location,
    n_features,
    step_size
)

In [14]:
import numpy as np

n_examples = len(indices)

x = np.ndarray(shape=(n_examples, n_features), dtype=np.float32)
y = np.ndarray(shape=(n_examples), dtype=np.float32)

pickup_hours = []

for i, idx in enumerate(indices):
    x[i] = ts_data_one_location.iloc[idx[0]:idx[1]]['rides'].values
    
    y[i] = ts_data_one_location.iloc[idx[1]]['rides']   
    
    pickup_hours.append(
        ts_data_one_location.iloc[idx[1]]['pickup_hour']
    )

In [15]:
print(f'{x.shape=}')
print(f'{x[:2]=}')
print(f'{pickup_hours[:5]=}')

x.shape=(719, 24)
x[:2]=array([[ 93.,  81.,  30.,  15.,   4.,   4.,   4.,  12.,  12.,  23.,  37.,
         41., 103.,  97., 106., 120., 104.,  65.,  39.,  35.,  32.,  41.,
         18.,  13.],
       [ 81.,  30.,  15.,   4.,   4.,   4.,  12.,  12.,  23.,  37.,  41.,
        103.,  97., 106., 120., 104.,  65.,  39.,  35.,  32.,  41.,  18.,
         13.,   2.]], dtype=float32)
pickup_hours[:5]=[Timestamp('2023-01-02 00:00:00'), Timestamp('2023-01-02 01:00:00'), Timestamp('2023-01-02 02:00:00'), Timestamp('2023-01-02 03:00:00'), Timestamp('2023-01-02 04:00:00')]


In [16]:
features_one_location = pd.DataFrame(
    x,
    columns=[f'rides_previous_{i+1}_hour' for i in reversed(range(n_features))]
)

features_one_location.head()

,rides_previous_24_hour,rides_previous_23_hour,rides_previous_22_hour,rides_previous_21_hour,rides_previous_20_hour,rides_previous_19_hour,rides_previous_18_hour,rides_previous_17_hour,rides_previous_16_hour,rides_previous_15_hour,...,rides_previous_10_hour,rides_previous_9_hour,rides_previous_8_hour,rides_previous_7_hour,rides_previous_6_hour,rides_previous_5_hour,rides_previous_4_hour,rides_previous_3_hour,rides_previous_2_hour,rides_previous_1_hour
0,93.0,81.0,30.0,15.0,4.0,4.0,4.0,12.0,12.0,23.0,...,106.0,120.0,104.0,65.0,39.0,35.0,32.0,41.0,18.0,13.0
1,81.0,30.0,15.0,4.0,4.0,4.0,12.0,12.0,23.0,37.0,...,120.0,104.0,65.0,39.0,35.0,32.0,41.0,18.0,13.0,2.0
2,30.0,15.0,4.0,4.0,4.0,12.0,12.0,23.0,37.0,41.0,...,104.0,65.0,39.0,35.0,32.0,41.0,18.0,13.0,2.0,0.0
3,15.0,4.0,4.0,4.0,12.0,12.0,23.0,37.0,41.0,103.0,...,65.0,39.0,35.0,32.0,41.0,18.0,13.0,2.0,0.0,2.0
4,4.0,4.0,4.0,12.0,12.0,23.0,37.0,41.0,103.0,97.0,...,39.0,35.0,32.0,41.0,18.0,13.0,2.0,0.0,2.0,2.0


In [17]:
targets_one_location = pd.DataFrame(
    y,
    columns=['target_rides_next_hour']
)

targets_one_location.head()

,target_rides_next_hour
0,2.0
1,0.0
2,2.0
3,2.0
4,0.0


In [18]:
final_dataset = pd.concat(
    [features_one_location, targets_one_location],
    axis=1
)

final_dataset.head()

,rides_previous_24_hour,rides_previous_23_hour,rides_previous_22_hour,rides_previous_21_hour,rides_previous_20_hour,rides_previous_19_hour,rides_previous_18_hour,rides_previous_17_hour,rides_previous_16_hour,rides_previous_15_hour,...,rides_previous_9_hour,rides_previous_8_hour,rides_previous_7_hour,rides_previous_6_hour,rides_previous_5_hour,rides_previous_4_hour,rides_previous_3_hour,rides_previous_2_hour,rides_previous_1_hour,target_rides_next_hour
0,93.0,81.0,30.0,15.0,4.0,4.0,4.0,12.0,12.0,23.0,...,120.0,104.0,65.0,39.0,35.0,32.0,41.0,18.0,13.0,2.0
1,81.0,30.0,15.0,4.0,4.0,4.0,12.0,12.0,23.0,37.0,...,104.0,65.0,39.0,35.0,32.0,41.0,18.0,13.0,2.0,0.0
2,30.0,15.0,4.0,4.0,4.0,12.0,12.0,23.0,37.0,41.0,...,65.0,39.0,35.0,32.0,41.0,18.0,13.0,2.0,0.0,2.0
3,15.0,4.0,4.0,4.0,12.0,12.0,23.0,37.0,41.0,103.0,...,39.0,35.0,32.0,41.0,18.0,13.0,2.0,0.0,2.0,2.0
4,4.0,4.0,4.0,12.0,12.0,23.0,37.0,41.0,103.0,97.0,...,35.0,32.0,41.0,18.0,13.0,2.0,0.0,2.0,2.0,0.0


In [19]:
from tqdm import tqdm
import numpy as np
import pandas as pd

def transform_ts_data_into_features_and_target(
    ts_data: pd.DataFrame,
    input_seq_len: int,
    step_size: int
):
    location_ids = ts_data['user_location_id'].unique()
    
    features = pd.DataFrame()
    targets = pd.DataFrame()
    
    for location_id in tqdm(location_ids):
        
        ts_data_one_location = ts_data.loc[
            ts_data.user_location_id == location_id,
            ['pickup_hour', 'rides']
        ].reset_index(drop=True)
        
        indices = get_cutoff_indices(
            ts_data_one_location,
            input_seq_len,
            step_size
        )
        
        n_examples = len(indices)
        
        x = np.ndarray((n_examples, input_seq_len), dtype=np.float32)
        y = np.ndarray((n_examples), dtype=np.float32)
        
        pickup_hours = []
        
        for i, idx in enumerate(indices):
            x[i] = ts_data_one_location.iloc[idx[0]:idx[1]]['rides'].values
            y[i] = ts_data_one_location.iloc[idx[1]]['rides']   # important fix
            
            pickup_hours.append(
                ts_data_one_location.iloc[idx[1]]['pickup_hour']
            )
        
        features_one_location = pd.DataFrame(
            x,
            columns=[f'rides_previous_{i+1}_hour' for i in reversed(range(input_seq_len))]
        )
        
        features_one_location['pickup_hour'] = pickup_hours
        features_one_location['user_location_id'] = location_id
        
        targets_one_location = pd.DataFrame(
            y,
            columns=['target_rides_next_hour']
        )
        
        features = pd.concat([features, features_one_location])
        targets = pd.concat([targets, targets_one_location])
    
    features.reset_index(drop=True, inplace=True)
    targets.reset_index(drop=True, inplace=True)
    
    return features, targets['target_rides_next_hour']

In [20]:
features, targets = transform_ts_data_into_features_and_target(
    ts_data,
    input_seq_len=24*7+1,
    step_size=24,
)

100%|██████████| 257/257 [00:01<00:00, 139.47it/s]


In [21]:
print(features.shape)
print(targets.shape)

features.head()
targets.head()

(6168, 171)
(6168,)


0    37.0
1     1.0
2     1.0
3     1.0
4     1.0
Name: target_rides_next_hour, dtype: float32

In [22]:
targets.head()

0    37.0
1     1.0
2     1.0
3     1.0
4     1.0
Name: target_rides_next_hour, dtype: float32

In [23]:
features.iloc[0]
targets.iloc[0]

np.float32(37.0)

In [24]:
features.isna().sum().sum()

np.int64(0)

In [25]:
ts_data.columns

Index(['pickup_hour', 'rides', 'user_location_id'], dtype='str')